# Lesson 1: Agentic RAG - Code Along 4 - Agentic RAG

Now make the RAG agentic.
Before, we used a fixed pipeline.
Now we implement the search as a tool, which the LLM agent can call.
For this, we build upon the persistent RAG pipeline from before.

In [1]:
# dependencies
import json

from sqlitesearch import TextSearchIndex
from rag_helper import RAGBase
from anthropic import Anthropic

In [2]:
# define search tool
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [3]:
# build definition for the search tool so the LLM knows about and how to use it
search_tool = {
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [4]:
# connect to db and load data
index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [5]:
# initialize anthropic client for LLM API calls
anthropic_client = Anthropic()

In [6]:
# system prompt
INSTRUCTIONS = '''
Your task is to answer the course participants' questions about the course.

Retrieve context from the course's FAQ knowledge base using the search tool.
Use the context to find relevant information and provide accurate answers.
Answer only based on the information in the knowledge base.
Do not answer if the question is not already answered in the knowledge base.

You may optimize the search terms in order to increase the chance of finding
relevant results.
If the search does not yield any results, you may repeat the search up to two
times with optimized queries and related terms.

If also repeated search does not yield any relevant information and the answer
is not found in the context, respond with "I don't know."
Do not make up things!
'''

# initialize message history - allows appending newer ones
messages = [
    {
        "role": "user",
        "content": "I just discovered the course. Can I join it?"
    }
]

In [7]:
# query the LLM on the question and provide tool
response = anthropic_client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1024,
    system=INSTRUCTIONS,
    messages=messages,
    tools=[search_tool],
)

In [8]:
# print full response
for item in response:
    print(item)

('id', 'msg_01W2sqMZC7baHMAMEvFhxqoZ')
('container', None)
('content', [ToolUseBlock(id='toolu_01QT7VE1tFPgGyqGqt7p7bwg', caller=DirectCaller(type='direct'), input={'query': 'how to join the course enrollment registration'}, name='search', type='tool_use')])
('model', 'claude-haiku-4-5-20251001')
('role', 'assistant')
('stop_details', None)
('stop_reason', 'tool_use')
('stop_sequence', None)
('type', 'message')
('usage', Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=748, output_tokens=58, output_tokens_details=None, server_tool_use=None, service_tier='standard'))


In [9]:
# see that currently there is only one message
for message in messages:
    print(messages)

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}]


In [10]:
# append the model's resonse to the message history to enable agentic behavrior
messages.append({"role": "assistant", "content": response.content})
print("Appended tool use to message history.")

Appended tool use to message history.


In [11]:
# print message history to see tool use was added
for message in messages:
    print(message)

{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
{'role': 'assistant', 'content': [ToolUseBlock(id='toolu_01QT7VE1tFPgGyqGqt7p7bwg', caller=DirectCaller(type='direct'), input={'query': 'how to join the course enrollment registration'}, name='search', type='tool_use')]}


In [12]:
# call the tool

# initialize empty list to contain tool results
tool_results = []

# iterate through all tool call and get results
# here, this is actually not strictly necessary, because there is just one call
# but in more complex settings the agent could also have made two or more calls
# in cases like that, this loop will get a response for each one
for block in response.content:
    if block.type == "tool_use" and block.name == "search":
        results = search(**block.input)
        tool_results.append({
            "type": "tool_result",
            "tool_use_id": block.id,
            "content": json.dumps(results),
        })

# check what this produced
# print dictionary in JSON format for better readability
for result in tool_results:
    print(json.dumps(result, indent=2))

{
  "type": "tool_result",
  "tool_use_id": "toolu_01QT7VE1tFPgGyqGqt7p7bwg",
  "content": "[{\"id\": \"74eb249bbf\", \"course\": \"llm-zoomcamp\", \"section\": \"General Course-Related Questions\", \"question\": \"I just discovered the course. Can I still join?\", \"answer\": \"Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions.\"}, {\"id\": \"977bf7786c\", \"course\": \"llm-zoomcamp\", \"section\": \"General Course-Related Questions\", \"question\": \"Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\", \"answer\": \"You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\"}, {\"id\": \"bd31146b0e\", \"course\": \"llm-zoomcamp\", \"section\": \"General Course-Related Quest

In [13]:
# add the result to the message history
messages.append({"role": "user", "content": tool_results})

# check new history
for message in messages:
    print(message)

{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
{'role': 'assistant', 'content': [ToolUseBlock(id='toolu_01QT7VE1tFPgGyqGqt7p7bwg', caller=DirectCaller(type='direct'), input={'query': 'how to join the course enrollment registration'}, name='search', type='tool_use')]}
{'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_01QT7VE1tFPgGyqGqt7p7bwg', 'content': '[{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}, {"id": "977bf7786c", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?", "answer": "You don\'t need it. You\'re accepted. You can also just start learning

In [14]:
# query the LLM again with tool/search result in messages
response = anthropic_client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1024,
    system=INSTRUCTIONS,
    messages=messages,
    tools=[search_tool],
)

In [15]:
# print final response
print(response.content[0].text)

Great question! **Yes, you can join the course!** 

According to the course FAQ, you can start learning and submitting homework even if you just discovered it. Here are the key points:

- **You can join anytime** - you don't need to wait for formal confirmation. You can simply start learning and submitting homework while the submission form is open.
- **No official registration needed** - Registration is optional and just used to gauge interest. It's not checked against any registered list, so you can begin without it.

However, there's one important thing to note: **If you want to receive a certificate, you need to submit your project while the course is still accepting submissions.** This is because certificates are only awarded during live cohort runs when peer-review happens.

To get started, I recommend checking out:
- The [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/)
- The [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp)
- T

### Experiment with History

I want to see what happens when we send only parts of the history.
Like for example I want to try omitting the user question and see if the API
even accepts this, and if the model is confused then because it lacks context.

In [16]:
# exclude the user question and start with tool call right away
messages[1:3]

[{'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01QT7VE1tFPgGyqGqt7p7bwg', caller=DirectCaller(type='direct'), input={'query': 'how to join the course enrollment registration'}, name='search', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01QT7VE1tFPgGyqGqt7p7bwg',
    'content': '[{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}, {"id": "977bf7786c", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?", "answer": "You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without reg

The assistant's search query is still rather conclusive about the user's intend.
My guess is that it will still answer to the question.
I assume that it will even provide a rather nice answer, because there's also
still the system prompt.
But let's see if the API reguses to accept this.

In [17]:
# query on manipulated message history and print result right away
response = anthropic_client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1024,
    system=INSTRUCTIONS,
    messages=messages[1:3],
    tools=[search_tool],
)
print(response.content[0].text)

Hello! I'm here to help answer your questions about the course. I can see there are several topics covered in the course FAQ, including:

- **General enrollment and registration** - You can join the course at any time, and don't need a confirmation email to get started
- **Certificates** - Information about how to earn a certificate
- **Course structure** - How to follow the weekly workflow and get started
- **Course scheduling** - When the next course will be offered

Please feel free to ask me any specific questions you have about the course, and I'll search the knowledge base to find the answers for you!


Interesting! Very similar to what I guessed, but still a little different.
The model still provided the information answering the question, but it did not
assume the user asked about this particular thing.
This was Haiku.
Let's see how Sonnet and Opus behave.
Perhaps they will be able to guess the user's question.
If they don't then this seems to be a general pattern of behavior.

In [18]:
# try sonnet 4.5, which is the second to latest release
response = anthropic_client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=1024,
    system=INSTRUCTIONS,
    messages=messages[1:3],
    tools=[search_tool],
)
print(response.content[0].text)

Hello! How can I help you with the course today? I have access to the course FAQ and can answer questions about enrollment, course materials, certificates, homework, and other course-related topics.


In [19]:
# now try the latest opus, 4.8
# this is the strongest model except for fable and mythos, which are banned :D
response = anthropic_client.messages.create(
    model="claude-opus-4-8",
    max_tokens=1024,
    system=INSTRUCTIONS,
    messages=messages[1:3],
    tools=[search_tool],
)
print(response.content[0].text)

Hello! I'd be happy to help you with questions about the course. Could you let me know what specifically you'd like to know?

In the meantime, here's some helpful information I found:

- **Can you still join?** Yes! Even if you just discovered the course, you can still join. If you want a certificate, just make sure to submit your project while submissions are still being accepted.

- **Do you need to register?** No registration is strictly required. You're accepted automatically, and you can start learning and submitting homework right away (while the form is open). Registration just helps gauge interest before the start date.

Feel free to ask me anything specific about the course!


Okay they all behave similarly.
I assume this (also) is because of the use system prompt.
Let's just omit the system prompt and see if that works.

In [21]:
# query on manipulated message history and print result right away
response = anthropic_client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1024,
    #system=INSTRUCTIONS,
    messages=messages[1:3],
    tools=[search_tool],
)
print(response.content[0].text)

Hello! I found some information about joining the course. Here's what you should know:

**Can I still join?**
Yes, you can still join the course anytime! However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.

**Do I need to register?**
Registration is optional. You don't need a confirmation email to start learning. You can begin immediately and submit homework without officially registering - registration is mainly just to gauge interest before the course starts.

**How do I get started?**
1. Check out the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/) and [GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp)
2. Visit the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/) to see deadlines

**Typical workflow:**
1. Watch the lesson videos
2. Work through lesson notebooks/code
3. Read homework instructions on GitHub
4. Submit answers through the course pl

Yes, removing the system prompt actually removed this particular behavior.
It specifically mentioned the user's questions.
Without it, it just summarizes the information and later asks if there was
anything specific the user wanted to know.

Just for fun, now let's send **only** the tool results and nothing else.
Not even the assistant's own decision to make that call in the first place.

In [34]:
# slicing like this gets us the tool results only
messages[2:3]

[{'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01QT7VE1tFPgGyqGqt7p7bwg',
    'content': '[{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}, {"id": "977bf7786c", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?", "answer": "You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."}, {"id": "bd31146b0e", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question

In [35]:
# send ONLY the tool results
try:
    response = anthropic_client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=1024,
        #system=INSTRUCTIONS,
        messages=messages[2:3],
        tools=[search_tool],
    )
    print(response.content[0].text)
except Exception as e:
    print(e)

Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.0.content.0: unexpected `tool_use_id` found in `tool_result` blocks: toolu_01QT7VE1tFPgGyqGqt7p7bwg. Each `tool_result` block must have a corresponding `tool_use` block in the previous message.'}, 'request_id': 'req_011CcGTHfNZfPoGjsR34KW9p'}


The API doesn't let us do this.
Every tool result block needs to have a tool call block before it.